# 中医药命名实体识别：Qwen2.5-7B + LoRA / QLoRA

这个 Notebook 与项目脚本使用同一套数据转换和词级评测代码，适合逐格执行并理解流程。

**目标**：把字符级 BIO 标注转换成对话式 SFT 数据，训练可替换的 Qwen2.5-7B adapter，最后按完整实体词而不是单字打印十种类别的 F1。

## 0. 准备路径与依赖

建议从项目根目录启动 `jupyter lab`。下面代码同时兼容 Notebook 当前目录为项目根目录或 `notebooks` 子目录的情况。

In [ ]:
# 导入 sys，用来把项目根目录加入 Python 模块查找路径。
import sys
# 导入 subprocess，后续用它执行预处理、训练和评价脚本。
import subprocess
# 导入 Path，以跨平台方式处理文件夹路径。
from pathlib import Path

# 如果当前在 notebooks 目录中，就将其父目录认作项目根目录。
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# 将根目录加入查找路径，以便导入 ner_utils.py。
if str(PROJECT_ROOT) not in sys.path:
    # 插到最前面，确保读取的是当前教学项目里的工具文件。
    sys.path.insert(0, str(PROJECT_ROOT))
# 打印根目录，确认之后读写的是正确工程。
print("项目根目录：", PROJECT_ROOT)

In [ ]:
# 首次配置环境时，请先按本机 CUDA 版本安装 PyTorch。
# 然后取消下一行开头的 #，利用上方确定的项目路径安装其余依赖。
# subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)

# 注意：Qwen2.5-7B 的正式训练需要 NVIDIA GPU；CPU 只适合检查数据处理步骤。
print("依赖安装完成后，再继续执行下面的数据单元格。")

## 1. 读取原始 BIO 标注

这个数据集的任务不是生成诊断，而是在病例、方剂或疗效描述片段中圈出有意义的词并分类。三个文件的作用是：`medical.train` 用于学习，`medical.dev` 用于训练期间验证，`medical.test` 只用于最终评分。

原始文件每行包含一个字符和一个标签，空行分隔两段文本：

| 标签 | 意思 | 示例如何阅读 |
| --- | --- | --- |
| `B-类别` | 一个实体词从这个字开始 | `当 B-中药` 表示中药实体从“当”开始 |
| `I-类别` | 继续属于前面开始的同类实体 | `归 I-中药` 与“当”组成“当归” |
| `O` | 这个字不是需要抽取的实体 | 标点、普通叙述文字常为 `O` |

例如 `活 B-方剂 / 络 I-方剂 / 效 I-方剂 / 灵 I-方剂 / 丹 I-方剂` 会拼成完整实体 `活络效灵丹`，类别为 `方剂`。工具函数也会保留正文中的全角空格，因为删掉它会导致实体字符位置偏移。

更完整的十种类别解释与真实样例见项目根目录的 `DATASET_GUIDE.md`。

In [ ]:
# 从工具文件导入固定类别列表和 BIO 读取函数。
from ner_utils import ENTITY_TYPES, bio_to_entities, read_bio_sentences

# 读取训练集，并自动按空行切分句子。
train_sentences = read_bio_sentences(PROJECT_ROOT / "data" / "medical.train")
# 选择第一句话作为可视化示例。
first_sentence = train_sentences[0]
# 拼接逐字输入，还原自然语言文本。
first_text = "".join(first_sentence["chars"])
# 将 BIO 标签恢复成完整实体词及其字符跨度。
first_entities = bio_to_entities(first_sentence["chars"], first_sentence["labels"])
# 显示总句数，检查原始训练数据读取是否成功。
print("训练集句子数：", len(train_sentences))
# 显示固定允许的十种实体类别。
print("实体类别：", ENTITY_TYPES)
# 显示第一条原文。
print("第一条文本：", first_text)
# 显示由逐字标注恢复得到的完整实体。
print("第一条实体：", first_entities)

### 1.1 用真实处方片段练习读标签

验证集第一条样本中包含一个方剂名和多味中药。下面同时打印“逐字标签”和“拼回后的完整实体”，可以直观看出 BIO 如何工作。

In [ ]:
# 读取验证集，选择包含“活络效灵丹”和中药名称的第一条样本。
dev_sentences = read_bio_sentences(PROJECT_ROOT / "data" / "medical.dev")
# 取出第一条验证样本。
example_sentence = dev_sentences[0]
# 将逐字内容拼回可以阅读的完整原文。
example_text = "".join(example_sentence["chars"])
# 将 BIO 标注恢复成真正参与词级评分的实体列表。
example_entities = bio_to_entities(example_sentence["chars"], example_sentence["labels"])
# 打印完整原文，让读者先知道这段话说了什么。
print("还原后的原文：", example_text)
# 打印标题，用来区分下面的逐字观察结果。
print("\n前 14 个字符的逐字标注：")
# 将字符与标签配对，只展示足以看懂方剂和前两味中药的开头部分。
for char, label in zip(example_sentence["chars"][:14], example_sentence["labels"][:14]):
    # 每行展示一个字符对应的 BIO 标签，与原始文件格式保持一致。
    print(f"{char}  {label}")
# 打印由程序拼出的完整实体词，观察 B 与 I 的组合结果。
print("\n还原出的完整实体：")
# 逐一显示实体词、类别和字符跨度。
for entity in example_entities:
    # 使用容易阅读的句子形式显示一个实体。
    print(entity["entity"], "->", entity["type"], f"[{entity['start']}:{entity['end']}]")

### 1.2 十种实体类别速查表

| 类别 | 通俗理解 | 数据中的示例 |
| --- | --- | --- |
| `中医治则` | 治疗原则或治法思想 | 疏肝理气、清热解毒 |
| `中医治疗` | 中医具体治疗方式 | 针灸、热敏灸 |
| `中医证候` | 辨证证型 | 肝郁脾虚、寒湿困脾 |
| `中医诊断` | 中医疾病诊断 | 泄泻、阳黄 |
| `中药` | 单味中药名称 | 当归、丹参 |
| `临床表现` | 症状或体征 | 口苦、腹痛 |
| `其他治疗` | 其他辅助干预 | 心理疏导、日光浴 |
| `方剂` | 中医复方名称 | 活络效灵丹、五苓散 |
| `西医治疗` | 西药或现代治疗 | 法莫替丁、莫沙必利 |
| `西医诊断` | 西医诊断名称 | 肺结核、功能性消化不良 |

做标注任务时以数据集中的标签为准。例如只命中 `腹痛` 中的 `痛` 并不算识别成功，因为评价的是完整实体词。

## 2. 转换为 SFT 数据

训练答案采用 JSON 数组，例如 `[{"entity":"口苦","type":"临床表现","start":3,"end":5}]`。其中 `end` 是右开区间，满足 `text[start:end] == entity`。

In [ ]:
# 使用前面导入的 subprocess 调用已经详细注释过的预处理脚本。
# 组合预处理命令，明确输入 data 文件夹和输出 processed_data 文件夹。
prepare_command = [
    sys.executable,
    str(PROJECT_ROOT / "prepare_sft_data.py"),
    "--data_dir",
    str(PROJECT_ROOT / "data"),
    "--output_dir",
    str(PROJECT_ROOT / "processed_data"),
]
# 在项目根目录执行命令，保证配置中的相对路径含义一致。
subprocess.run(prepare_command, cwd=PROJECT_ROOT, check=True)

In [ ]:
# 导入 JSON，用漂亮的缩进格式展示处理后的训练记录。
import json
# 导入 JSONL 读取函数，加载已经生成的 SFT 文件。
from ner_utils import read_jsonl

# 读取统计摘要，检查三份数据的样本数量与实体分布。
summary = json.loads((PROJECT_ROOT / "processed_data" / "summary.json").read_text(encoding="utf-8"))
# 逐个数据划分打印样本数量和最长文本字符数。
for split_name, split_summary in summary.items():
    # 打印当前划分的关键规模指标。
    print(split_name, split_summary["num_samples"], split_summary["max_text_characters"])
# 读取 SFT 训练集记录。
processed_train = read_jsonl(PROJECT_ROOT / "processed_data" / "train.jsonl")
# 以缩进 JSON 展示第一条 prompt 与 completion。
print(json.dumps(processed_train[0], ensure_ascii=False, indent=2))

## 3. 选择模型与 LoRA / QLoRA

- `qlora`：4-bit NF4 加载基座模型，更省显存，适合作为首次实验选择。
- `lora`：以 BF16/FP16 加载基座模型，显存需求更高。

如需换模型，只修改 `MODEL_NAME`；换为不同模型架构时还应检查聊天模板、结束 token 和配置中的 `target_modules`。

In [ ]:
# 在 "qlora" 与 "lora" 之间二选一。
METHOD = "qlora"
# 默认选择适合指令式 SFT 的 Qwen2.5-7B-Instruct。
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
# 根据方法自动选择对应的 YAML 配置。
CONFIG_PATH = PROJECT_ROOT / "configs" / f"{METHOD}.yaml"
# 为本次 adapter 结果建立清晰的输出目录。
ADAPTER_PATH = PROJECT_ROOT / "outputs" / f"qwen2_5_7b_{METHOD}"
# 打印选择结果，执行训练前再核对一次。
print("方法：", METHOD)
# 打印基座模型名称，确保模型替换已经生效。
print("模型：", MODEL_NAME)
# 打印配置与输出路径。
print("配置：", CONFIG_PATH, "\n输出：", ADAPTER_PATH)

In [ ]:
# 导入 PyTorch，用来确认训练环境能否看到 CUDA GPU。
import torch
# 打印 CUDA 是否可用。
print("CUDA 可用：", torch.cuda.is_available())
# 如果有 GPU，则打印设备名称帮助核对运行机器。
if torch.cuda.is_available():
    # 读取第 0 张 GPU 的名称。
    print("GPU：", torch.cuda.get_device_name(0))
# 若没有 GPU，则提醒不要直接等待 7B 正式训练。
else:
    # CPU 下仍可执行前面的数据步骤和后面的评分单元测试。
    print("当前未检测到 CUDA；请在 GPU 环境中运行训练单元格。")

## 4. 开始 SFT 训练

下面单元格会真正下载或读取 7B 模型并训练 adapter。首次运行前请确认 GPU、磁盘空间与模型下载权限已经准备好。训练代码的每个步骤均在项目根目录的 `train_sft.py` 中配有中文注释。

In [ ]:
# 组合训练命令，并把当前 Notebook 中选择的模型和方法传给脚本。
train_command = [
    sys.executable,
    str(PROJECT_ROOT / "train_sft.py"),
    "--config",
    str(CONFIG_PATH),
    "--method",
    METHOD,
    "--model_name_or_path",
    MODEL_NAME,
    "--output_dir",
    str(ADAPTER_PATH),
]
# 打印即将运行的命令，方便保存到实验记录中。
print("即将执行：", " ".join(train_command))
# 正式运行训练；如显存不足，可回到配置文件降低批大小。
subprocess.run(train_command, cwd=PROJECT_ROOT, check=True)

## 5. 生成测试结果并计算完整实体 F1

评测要求预测实体的 `type`、`start`、`end`、`entity` 全部与人工标注一致。只猜中实体中的一个字不会得分。

In [ ]:
# 为当前方法指定预测明细文件。
PREDICTION_FILE = PROJECT_ROOT / "outputs" / f"{METHOD}_test_predictions.jsonl"
# 为当前方法指定结构化指标文件。
METRICS_FILE = PROJECT_ROOT / "outputs" / f"{METHOD}_test_metrics.json"
# 组合最终测试命令，读取刚刚训练得到的 adapter。
evaluate_command = [
    sys.executable,
    str(PROJECT_ROOT / "evaluate_ner.py"),
    "--config",
    str(CONFIG_PATH),
    "--adapter_path",
    str(ADAPTER_PATH),
    "--model_name_or_path",
    MODEL_NAME,
    "--method",
    METHOD,
    "--prediction_file",
    str(PREDICTION_FILE),
    "--metrics_file",
    str(METRICS_FILE),
]
# 正式生成测试集预测，并在输出中打印每一类实体 F1。
subprocess.run(evaluate_command, cwd=PROJECT_ROOT, check=True)

In [ ]:
# 读取刚刚保存的指标 JSON 文件，便于在 Notebook 中继续画图或分析错误。
metrics = json.loads(METRICS_FILE.read_text(encoding="utf-8"))
# 导入表格排版函数，以与终端脚本相同的口径再次展示结果。
from ner_utils import format_metric_report
# 打印全部十种类别的完整实体词级指标及总体结果。
print(format_metric_report(metrics))

## 6. 下一步可选优化

请阅读项目根目录下的 `INDUSTRIAL_OPTIMIZATION.md`。优先值得判断的项目包括：数据泄漏与冲突审计、小类别增强、验证集错误切片、结构化约束解码，以及与判别式 NER 基线的质量和时延对比。